In [ ]:
! pip install -r requirements.txt


In [5]:
# !pip install pinecone==8.1.0
# ! pip install langchain_groq

In [3]:
import os
from dotenv import load_dotenv, find_dotenv

# loading the API Keys from .env
load_dotenv(find_dotenv(), override=True)

# os.environ.get('OPENAI_API_KEY')

True

In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1
)

output = llm.invoke(
    "Explain quantum mechanics in one sentence."
)

print(output.content)

Quantum mechanics is the branch of physics that describes how particles behave at the smallest scales, where they exhibit wave‑particle duality, superposition, and inherently probabilistic outcomes rather than deterministic ones.


In [5]:
# authenticating to Pinecone. 
# the API KEY is in .env
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
Pine_cone_api_key = os.getenv("PINE_CONE_KEY")
print(Pine_cone_api_key)

pcsk_46wBxC_LLd4ardu7NgKkN3NSE7FCUNtT2vRMCEkVfU3gNTwnJLtT4Nt3zCD3o64GCjYwnq


In [ ]:
from pinecone import Pinecone, ServerlessSpec

# Initilizing and authenticating the pinecone client
pc = Pinecone(api_key=Pine_cone_api_key)
# pc = Pinecone(api_key='YOUR_API_KEY')

pc.create_index(
    name="langchain-index",
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)
# checking authentication
# pc.list_indexes()
indexes = pc.list_indexes()

print(indexes)
print(indexes.names())

In [9]:
pc.list_indexes()[0]['name']

'langchain-index'

In [10]:
pc.describe_index('langchain-index')

{
    "name": "langchain-index",
    "metric": "cosine",
    "host": "langchain-index-ftzdvl8.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [11]:
pc.list_indexes().names()

['langchain-index']

In [24]:
index_name ='langchain-index'
if index_name not in pc.list_indexes().names():
    print(f"creating index {index_name}")
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ) 
    )
    print('Index created! 😊')
else:
    print(f'Index {index_name} already exists!')

Index langchain-index already exists!


In [30]:
index_name = 'langchain-index'
if index_name in pc.list_indexes().names():
    print(f'Deleting index {index_name}')
    pc.describe_index(index_name)
    print('Done')
else:
    print(f'Index {index_name} Doesnt exists')

Deleting index langchain-index
Done


In [31]:
index_name = 'langchain-index'
index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3}},
 'total_vector_count': 3,
 'vector_type': 'dense'}

## Working with Vectors

In [ ]:
# inserting vectors
import random
vectors = [[random.random() for _ in range(1536)] for v in range(5)]
# print(vectors)
ids = list('abcde')
index_name = 'langchain-index'
index = pc.Index(index_name)

index.upsert(vectors=zip(ids, vectors))

[[0.43364995665319894, 0.6014014735833553, 0.2607517752358379, 0.019742832000983035, 0.023233878115439865, 0.599035374698352, 0.26496220447019814, 0.19490394596261484, 0.757220621051924, 0.8731868216086002, 0.304604706718549, 0.6560218845598071, 0.03265810044701567, 0.978380682094316, 0.1671258878055072, 0.38400067429140816, 0.972810910388176, 0.09269637586903079, 0.36107500204813014, 0.44845249038567203, 0.39632665501022535, 0.4781844186636245, 0.9897661181974238, 0.5420173968385474, 0.9113849916406996, 0.6046264795163202, 0.8735311426313592, 0.035362470733021456, 0.49365089341694113, 0.37470815251207124, 0.6273060672808448, 0.4420259873384377, 0.31370814260116986, 0.0343745464522478, 0.025990007326059827, 0.20150503246076534, 0.5660606081420848, 0.14271147124880468, 0.10109574724317916, 0.44317158599384954, 0.7281323834221807, 0.5692494850710581, 0.015026702975537432, 0.9457443955648959, 0.2309337467241156, 0.8768578528343299, 0.5423390902300231, 0.8883841231521502, 0.996925399957216

{'upserted_count': 5}

In [17]:
# updating vectors
index.upsert(vectors=[('c', [0.5] * 1536)])

{'upserted_count': 1}

In [18]:
# fetching vectors
# index = pc.Index(index_name)
index.fetch(ids=['c', 'd'])

FetchResponse(namespace='', vectors={'d': Vector(id='d', values=[0.943929136, 0.346416116, 0.476225466, 0.864412785, 0.569176197, 0.0557594076, 0.749959946, 0.766400635, 0.0928163677, 0.900917828, 0.403525621, 0.671309173, 0.867405713, 0.904972315, 0.535292864, 0.100332662, 0.722902358, 0.991066277, 0.564013958, 0.046252612, 0.382687777, 0.706095159, 0.992826164, 0.169180349, 0.236739039, 0.796755, 0.771811306, 0.776203811, 0.22639899, 0.72640717, 0.510596156, 0.960491896, 0.340887547, 0.679204524, 0.175485417, 0.575063, 0.640763283, 0.867874742, 0.881319046, 0.475466818, 0.733683109, 0.885516882, 0.118014283, 0.563940823, 0.132878616, 0.178735703, 0.234499946, 0.836636126, 0.432121664, 0.180624872, 0.575194597, 0.884201586, 0.81369704, 0.0134563707, 0.441857427, 0.0409731716, 0.67435205, 0.685682774, 0.0321420208, 0.173128694, 0.675630152, 0.247842148, 0.0200428367, 0.643302143, 0.59282732, 0.648103416, 0.0985756293, 0.27057758, 0.820486, 0.938818872, 0.372375607, 0.150842458, 0.11544

In [19]:
# deleting vectors
index.delete(ids=['b', 'c'])

{}

In [20]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3}},
 'total_vector_count': 3,
 'vector_type': 'dense'}

In [21]:
# querying a non-existing vector returns an empty vector
index.fetch(ids=['x']) 

FetchResponse(namespace='', vectors={}, usage={'egressBytes': 4.0, 'read_units': 1})

In [22]:
# querying vectors
query_vector = [random.random() for _ in range(1536)]

In [32]:
index.query(
    vector=query_vector,
    top_k=3,
    include_values=False
)

{'matches': [{'id': 'a', 'score': 0.75223273, 'values': []},
             {'id': 'd', 'score': 0.74938488, 'values': []},
             {'id': 'e', 'score': 0.746867836, 'values': []}],
 'namespace': '',
 'usage': {'egressBytes': 34.0, 'read_units': 1}}

## Namespaces

In [33]:
# index.describe_index_stats()
index = pc.Index('langchain-index')

import random
vectors = [[random.random() for _ in range(1536)] for v in range(5)]
ids = list('abcde')
index.upsert(vectors=zip(ids, vectors))

{'upserted_count': 5}

In [34]:
# partition the index into namespaces
# creating a new namespace
vectors = [[random.random() for _ in range(1536)] for v in range(3)]
ids = list('xyz')
index.upsert(vectors=zip(ids, vectors), namespace='first-namespace')

{'upserted_count': 3}

In [35]:
vectors = [[random.random() for _ in range(1536)] for v in range(2)]
ids = list('qp')
index.upsert(vectors=zip(ids, vectors), namespace='second-namespace')

{'upserted_count': 2}

In [36]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5},
                'first-namespace': {'vector_count': 3},
                'second-namespace': {'vector_count': 2}},
 'total_vector_count': 10,
 'vector_type': 'dense'}

In [37]:
index.fetch(ids=['x'])

FetchResponse(namespace='', vectors={}, usage={'egressBytes': 4.0, 'read_units': 1})

In [40]:
index.fetch(ids=['x'], namespace='first-namespace')


FetchResponse(namespace='first-namespace', vectors={}, usage={'egressBytes': 21.0, 'read_units': 1})

In [39]:
index.delete(ids=['x'], namespace='first-namespace')

{}

In [43]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5},
                'second-namespace': {'vector_count': 2}},
 'total_vector_count': 7,
 'vector_type': 'dense'}

In [42]:
index.delete(delete_all=True, namespace='first-namespace')

{}